# 12 - Reading the repair

**Purpose.** To explain what notebook `11` did to the constants published before session 04's
verdict, and what a reader should now believe about each of them. `11` is the notebook that *made*
these numbers, from frames already on disk, and is written for someone checking the work. This one
is written for someone deciding what to do next.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `state_repair_constants.json` and the predecessors it judges:
`ptc_constants.json`, `dark_constants.json`, `bias_constants.json`, `offset_state_constants.json`,
`dark_blocks.csv` and `offset_state_settings.csv`. Where arithmetic appears below it is done on
published numbers, to show what a published number is worth; if any of it disagreed with
`results/`, `results/` would be right and this notebook would be the bug.

**It assumes `00_statistics.ipynb`** for why a plane mean over a quarter of a million pixels
resolves a hundredth of a count, and why a pair difference measures a width and not a level. It
assumes `10` for the state itself: what H2 is, and why it reaches backwards.

**The headline: the state cost the model nothing, and cost the record one honest correction.**

1. **`g(gain)` barely moved.** Seven of the eight gains move less than session 02's own gain-law
   residual of 1.344%. Only gain 450 exceeds it, at +1.36%. The reason is algebra, not luck, and
   section 2 is that algebra: a mixture mean differenced against a mixture mean is *unbiased*. What
   the state injected into the PTC was scatter, not offset.
2. **The gain-100 bimodality is now published**, with provenance, through the classifier the record
   requires - 0.5129 counts where H2 predicted 0.4878, agreeing to +5.1%. It is out of sample in
   three ways, and it is the sharpest confirmation the H2 law has.
3. **It is not L31's mechanism.** Two independent arithmetic tests refute it, and both are in
   section 6. L31 goes to the linearity session narrowed, not drained.
4. **The dark bound was blamed on the wrong thing.** Session 03 said the offset state limited it.
   Its own published table refutes random hopping. What survives is an offset that tracks *exposure
   length* - which a peer-group rule cannot see, because exposure is what defines a peer group.
5. **None of this changes a capture setting.** Even the conditional dark current is 0.09% of L32's
   sky rate. The state is a calibration problem, as `10` said; this notebook is what it cost to
   confirm that in numbers rather than in words.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
read = lambda n: json.loads((RESULTS / n).read_text())

K = read("state_repair_constants.json")          # notebook 11, the repair
K1 = read("bias_constants.json")                 # session 01
K2 = read("ptc_constants.json")                  # session 02
K3 = read("dark_constants.json")                 # session 03
K4 = read("offset_state_constants.json")         # session 04

blocks = pd.read_csv(RESULTS / "dark_blocks.csv")
settings = pd.read_csv(RESULTS / "offset_state_settings.csv")
rungs = pd.read_csv(RESULTS / "ptc_rungs.csv")

GAINS = [0, 50, 100, 190, 200, 250, 300, 450]
HCG = K1["hcg_threshold_gain"]["value"]
PED_FIT = K1["pedestal_fit"]["value"]
ANCHOR = K3["offset_state_step"]["value"]        # 0.9931 counts at gain 250
RESIDUAL_PCT = K2["gain_law"]["value"]["residual_pct"]
SKY_E_PER_S = 1.594                              # L32, green, unfiltered, Bortle 5-6

num = lambda d: {int(k): v for k, v in d.items()}
G_PUB = num(K2["system_gain"]["value"])
G_NEW = num(K["system_gain_restated"]["value"])
MOVE = num(K["system_gain_move_pct"]["value"])
MIXOFF = num(K["ptc_pedestal_mixture_offset"]["value"])
OCC_BIAS = num(K["ptc_bias_occupancy"]["value"])


def h2_step(gain):
    """The H2 law, rebuilt from published constants exactly as 10 and 11 build it:
    session 01's analog pedestal term, normalised through session 03's anchor."""
    branch = PED_FIT["hcg" if gain >= HCG else "lcg"]
    at_anchor = PED_FIT["hcg"]["B"] * 10 ** (250 / 200)
    return ANCHOR * branch["B"] * 10 ** (gain / 200) / at_anchor


print("notebook 11 published %d constants on %s, from %d source frames"
      % (len(K), K["system_gain_restated"]["measured_on"],
         K["system_gain_restated"]["source_frames"]))
print("session 02's gain-law residual, the yardstick used throughout: %.3f%%" % RESIDUAL_PCT)
print("H2 step in counts:", {g: round(h2_step(g), 4) for g in GAINS})

---

## 1. What was actually wrong, in one paragraph

Session 02's PTC needs a signal axis. It built one as `mean_flat - pedestal`, where the pedestal
was the mean of ten bias frames taken once per gain. Session 04 then showed those ten frames were
not ten samples of one level: they were a **mixture of two states**, separated by anything from
0.16 counts at gain 0 to 10.31 at gain 450.

That makes the pedestal a mixture mean, and it makes the flats mixture means too. The PTC's
intercept was **fixed** at session 01's read noise rather than fitted, so every error in the signal
axis has nowhere to go but the slope - and the slope *is* `g`. That is why `g` was the exposed
constant and `R` was not: `R` is a width in space, and a level that is uniform across the frame
cancels out of it.

So the question `11` had to answer is not "is the pedestal wrong" - it obviously is - but **how
much of that wrongness survives the subtraction**.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.2))

g = np.linspace(0, 460, 400)
for lo, hi in ((0, HCG - 1), (HCG, 460)):
    seg = np.linspace(lo, hi, 200)
    ax[0].plot(seg, [h2_step(x) for x in seg], "-", lw=1.2, color="crimson",
               label="H2 step" if lo == 0 else None)
ax[0].plot(250, ANCHOR, "*", ms=12, color="tab:green", label="session 03 anchor")
ax[0].axvline(HCG, color="0.7", lw=0.8, ls=":")
ax[0].set(yscale="log", xlabel="gain", ylabel="step between states, ADC counts",
          title="how big the state is at each PTC gain")
ax[0].legend(fontsize=7)

lowest = rungs[rungs.usable == True].groupby("gain").signal.min()
ax[1].plot(lowest.index, lowest.values, "o-", ms=6, lw=1.0, color="0.35",
           label="lowest usable rung, counts")
ax[1].plot(GAINS, [h2_step(x) for x in GAINS], "s-", ms=6, lw=1.0, color="crimson",
           label="the state, counts")
ax[1].set(yscale="log", xlabel="gain", ylabel="ADC counts",
          title="the state against the smallest signal it contaminates")
ax[1].legend(fontsize=7)
plt.tight_layout()

print("lowest usable rung against the step, per gain:")
for x in GAINS:
    if x in lowest.index:
        print("  gain %3d   lowest rung %8.2f counts   step %7.3f   = %5.2f%% of it"
              % (x, lowest[x], h2_step(x), 100 * h2_step(x) / lowest[x]))

## 2. Why the answer is "not much": a mixture minus a mixture is unbiased

This is the piece that decides everything downstream, and it is three lines of algebra. Write `f_b`
for the fraction of the ten bias frames that sat in the far state, and `f_f` for the same fraction
among the four flats at one rung:

```
pedestal_used   = L_near + f_b * step
mean_flat(rung) = S_true + L_near + f_f * step
signal          = mean_flat - pedestal_used = S_true + (f_f - f_b) * step
```

The error is `(f_f - f_b) * step`, and **its expectation is zero** whenever flats and bias hop with
the same occupancy. A mixture mean is an unbiased estimate of a mixture mean. So the state did not
push `g` in a direction; it added **noise, rung by rung**.

**And that noise is small where it matters and large where it does not.** `f_f` is drawn from only
four frames, so at 25% occupancy its standard deviation is about 0.22. At gain 450 that is 2.2
counts on the flat side alone, and 2.5 once the ten bias frames are included - on a signal axis
whose lowest usable rungs are only a few counts. That sounds alarming, and it is exactly why gain
450 is the one gain that moved past the yardstick.

**The trap this rules out.** The obvious "fix" - replace the mixture-mean pedestal with the near
state and stop - removes `f_b * step` while leaving `f_f * step` in place. That does not remove a
bias; it *creates* one. `11` ran that fit deliberately, as its "bias-side only" column, and
published the gap between it and the correct two-sided fit as the uncertainty on the restated `g`.
That gap is the honest measure of what the drift-limited rungs leave open.

In [ ]:
occ = 0.25
n_flat, n_bias = 4, 10
sd_f = np.sqrt(occ * (1 - occ) / n_flat)
sd_b = np.sqrt(occ * (1 - occ) / n_bias)
sd_diff = np.sqrt(sd_f ** 2 + sd_b ** 2)

print("at %.0f%% occupancy:  sd(f_f) over %d flats = %.3f,  sd(f_b) over %d bias = %.3f"
      % (100 * occ, n_flat, sd_f, n_bias, sd_b))
print("so sd(f_f - f_b) = %.3f, and in counts that is:" % sd_diff)
for x in GAINS:
    print("  gain %3d   %.3f counts of scatter on the signal axis" % (x, sd_diff * h2_step(x)))

print()
print("what 11 measured on session 02's own bias groups (a third independent night):")
tab = pd.DataFrame({"gain": GAINS,
                    "far fraction f_b": [OCC_BIAS[x] for x in GAINS],
                    "mixture offset, counts": [MIXOFF[x] for x in GAINS],
                    "H2 step, counts": [round(h2_step(x), 4) for x in GAINS]})
print(tab.to_string(index=False))
print()
print("session 04 measured 0.333 at gain 0 and 0.250 at gain 450; notebook 10 backed")
print("0.200 out of session 01's published pedestal at gain 450.  Three nights, same order.")

## 3. Where the state was readable at all, and where the panel wins

Putting both sides in the same state means classifying the flats, and on a flat the classifier has
a rival: **the light panel drifts**. L31 records repeat-to-repeat spreads up to 1.79%, and panel
drift scales with the signal while the state is fixed in counts. So the state is only readable on a
flat where

```
step  >>  drift_frac * S        i.e.   S  <<  step / drift_frac
```

`11` set the margin at 3x and computed that boundary per gain rather than assuming it. The result
is that the state is readable on the **low rungs only** - and the low rungs are exactly where the
`1/var^2` weighting leans hardest, which is the one piece of luck in this whole story.

Note what this means for the shape of the correction: at gain 0 the step is 0.16 counts, the
readable boundary is a few counts, and essentially nothing is readable - but nothing needs to be,
because 0.16 counts is beneath everything that consumes it. At gain 450 the step is about 10
counts, the boundary is around 185 counts, and most of the low rungs are readable. **The correction
is available precisely where it is needed.**

In [ ]:
DRIFT_FRAC, MARGIN = 0.0179, 3.0          # L31's worst spread, and 11's margin

rows = []
for x in GAINS:
    bound = h2_step(x) / (MARGIN * DRIFT_FRAC)
    us = rungs[(rungs.gain == x) & (rungs.usable == True)]
    rows.append({"gain": x, "H2 step": round(h2_step(x), 3),
                 "readable below S =": round(bound, 1),
                 "usable plane-rungs": len(us),
                 "of those readable": int((us.signal < bound).sum())})
read_tab = pd.DataFrame(rows)
print(read_tab.to_string(index=False))
print()
print("total: %d of %d usable plane-rungs are readable for state; the rest are drift-limited"
      % (read_tab["of those readable"].sum(), read_tab["usable plane-rungs"].sum()))

fig, ax = plt.subplots(figsize=(6.4, 3.4))
us = rungs[rungs.usable == True]
ax.plot(us.gain + np.random.uniform(-4, 4, len(us)), us.signal, ".", ms=3,
        color="0.6", label="usable plane-rungs")
ax.plot(read_tab.gain, read_tab["readable below S ="], "s-", ms=6, lw=1.2,
        color="crimson", label="readable boundary: step / (3 x 1.79%)")
ax.set(yscale="log", xlabel="gain", ylabel="rung signal, ADC counts",
       title="a rung below the red line can be classified; above it the panel wins")
ax.legend(fontsize=7)
plt.tight_layout()

## 4. What `g` actually moved, and how to read the movement

The yardstick is session 02's own **gain-law residual, 1.344%**. That is how far the published
`g(gain)` points already scatter about the law fitted through them, so a move smaller than that is
inside noise the record already carries - it is not a correction anybody could act on.

**Seven of the eight gains are inside it.** The two worth naming:

- **gain 300, -1.16%.** Inside the yardstick, but the largest move that is. Its bias mixture offset
  is -0.37 counts, the biggest of any gain on the bias side.
- **gain 450, +1.36%.** The only gain past the yardstick, and the interesting one, because its bias
  side contributed **nothing** - `11` could not resolve two states in those ten bias frames at all.
  The entire move comes from the **flats**, where a 10-count step sits on rungs of only a few
  counts. That is section 2's scatter argument and section 3's readability argument meeting: the
  one gain where the state was big enough to matter is the one gain where it was also readable.

**What this does not license.** The restated numbers are not obviously better than the published
ones at the six gains where the move is a fraction of a percent - they are a second estimate from
the same frames, differing by less than the spread the frames already have. Which set the record
should carry is section 8's open decision, and it is deliberately not this notebook's to take.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.0, 3.4))

mv = [MOVE[x] for x in GAINS]
colours = ["crimson" if abs(m) > RESIDUAL_PCT else "0.45" for m in mv]
ax[0].bar([str(x) for x in GAINS], mv, color=colours)
ax[0].axhspan(-RESIDUAL_PCT, RESIDUAL_PCT, color="tab:green", alpha=0.15,
              label="session 02's own 1.344% gain-law residual")
ax[0].axhline(0, color="0.6", lw=0.6)
ax[0].set(xlabel="gain", ylabel="change in g, percent",
          title="what the state cost g")
ax[0].legend(fontsize=7)

ax[1].plot(GAINS, [G_PUB[x] for x in GAINS], "o-", ms=6, lw=1.0, color="tab:blue",
           label="session 02 published")
ax[1].plot(GAINS, [G_NEW[x] for x in GAINS], "s--", ms=5, lw=1.0, color="crimson",
           label="11 restated")
ax[1].set(yscale="log", xlabel="gain", ylabel="g, e- per ADC count",
          title="the two curves, on the axis the model uses")
ax[1].legend(fontsize=7)
plt.tight_layout()

unc = num(K["system_gain_restated"]["uncertainty"])
out = pd.DataFrame({"gain": GAINS,
                    "published": [G_PUB[x] for x in GAINS],
                    "restated": [G_NEW[x] for x in GAINS],
                    "move %": [MOVE[x] for x in GAINS],
                    "inside 1.344%": [abs(MOVE[x]) <= RESIDUAL_PCT for x in GAINS],
                    "bias-vs-both gap": [unc[x] for x in GAINS]})
print(out.to_string(index=False))
print()
print("inside the yardstick: %d of %d gains" % (int(out["inside 1.344%"].sum()), len(GAINS)))
worst = max(GAINS, key=lambda x: abs(MOVE[x]))
print("largest move: gain %d, %+.2f%% (%.5f -> %.5f e-/ADU)"
      % (worst, MOVE[worst], G_PUB[worst], G_NEW[worst]))
print("its bias-side mixture offset was %.4f counts - the move is the flats, not the bias"
      % MIXOFF[worst])

## 5. Gain 100: the prediction that was already sitting on disk

This is the best evidence the H2 law has, and it cost nothing to collect.

Session 04 could not measure the step at gain 100. H2 predicted 0.488 counts there and the night's
resolution limit was 0.776 - the classifier reported `None`, correctly, and `10` said so. But
session 01 had shot **450 bias frames at gain 100** a week earlier, to ask an entirely different
question: does the pedestal drift? Those frames are `results/pedestal_drift.csv`.

`11` ran the published classifier over that column: **0.5129 counts, against a prediction of
0.4878 - agreement to about +5%**, from a law normalised at gain 250 on a different night.

**Out of sample in three ways at once** - a different night, a different notebook, and a gain the
measuring night's own classifier could not resolve. The in-sample fit in session 04 *chose* H2 over
H1; this *predicted* an unobserved separation before anyone looked.

**And it obeys D71.** `10` reported this agreement as 0.6% using an edge-to-edge gap between the
two clusters, which is biased low by both tails. `stats.offset_state` estimates the separation
between cluster **centres**, which is the quantity H2 predicts, and it gives 5.1%. The published
number is the classifier's, not the gap's. A one-line estimator invented in a scratchpad is exactly
what the rule "nothing enters `results/` except through a notebook" exists to catch.

In [ ]:
sep100 = K["state_step_at_gain100"]["value"]
sc100 = K["state_step_at_gain100"]["uncertainty"]
pred100 = h2_step(100)

print("gain 100, from session 01's 450-frame drift block:")
print("  measured separation   %.4f counts  (+/- %.4f within-state scatter)" % (sep100, sc100))
print("  H2 predicted          %.4f counts  (%+.1f%%)"
      % (pred100, 100 * (sep100 / pred100 - 1)))
print("  H1 would have predicted %.4f counts (%+.0f%%)"
      % (ANCHOR, 100 * (sep100 / ANCHOR - 1)))
print("  separation over scatter: %.0f sigma - not a marginal call" % (sep100 / sc100))
print()
print("what session 04's own night could do at gain 100:")
s100 = settings[settings.gain == 100]
print(s100[["gain", "offset", "n", "separation", "scatter", "resolution_limit"]]
      .round(4).to_string(index=False))
print("resolution limit %.3f against a %.3f-count step: it could not have seen this."
      % (float(s100.resolution_limit.iloc[0]), pred100))

fig, ax = plt.subplots(figsize=(5.6, 3.2))
res = settings[settings.separation.notna()]
gg = np.linspace(0, 460, 400)
for lo, hi in ((0, HCG - 1), (HCG, 460)):
    seg = np.linspace(lo, hi, 200)
    ax.plot(seg, [h2_step(x) for x in seg], "-", lw=1.2, color="crimson",
            label="H2, no free parameters" if lo == 0 else None)
ax.plot(res.gain, res.separation, "o", ms=7, color="0.15", label="session 04, resolved")
ax.plot(250, ANCHOR, "*", ms=12, color="tab:green", label="session 03 anchor")
ax.plot(100, sep100, "D", ms=8, color="tab:blue", label="11, out of sample")
ax.set(yscale="log", xlabel="gain", ylabel="step, ADC counts",
       title="three sessions, one law, one point it did not fit")
ax.legend(fontsize=7)
plt.tight_layout()

## 6. The state is not L31, and the refutation is arithmetic

`10` offered the state as a named candidate for L31 - gain 100 not repeating while gain 200 did -
and was careful to call it a lead rather than a harvest. It is now testable from published numbers
alone, and it fails on two independent counts.

**The ratio test needs no knowledge of the rung levels at all.** L31 contrasts a 1.79% repeat
spread at gain 100 with 0.011% at gain 200: a factor of **163**. H2's steps at those two gains are
0.488 and 0.558 counts: a factor of **0.87**. The mechanism predicts the two gains behave *the
same*. It is not that the state is too small to explain L31 - it is that it points the wrong way.

**The magnitude test is independent of the first.** For a step to *be* 1.79% of a rung, that rung
would have to sit at about 27 counts. A linearity ladder runs 50-115% of saturation, which is
thousands of counts. At 2000 counts the step is 0.024% - two orders of magnitude short.

**So L31 keeps its question and loses a candidate.** What `11` *did* settle is the half of L31 that
is about the bimodality itself: that is now published with provenance, at the gain L31 complains
about. The entry goes to the linearity session narrowed, and `CLAUDE.md` is explicit that an entry
leaves the queue only when the claim it carries has been checked. This one has not.

In [ ]:
L31 = dict(gain100_spread_pct=1.79, gain200_spread_pct=0.011)
ratio_observed = L31["gain100_spread_pct"] / L31["gain200_spread_pct"]
ratio_state = h2_step(100) / h2_step(200)

print("verdict published by 11: state_explains_L31 = %s" % K["state_explains_L31"]["value"])
print()
print("test 1, the ratio:")
print("  L31's observed spread ratio, gain 100 : gain 200   %.0fx" % ratio_observed)
print("  the ratio H2 predicts (steps %.3f and %.3f counts)  %.2fx"
      % (h2_step(100), h2_step(200), ratio_state))
print("  the mechanism says the two gains behave the same; L31 says they differ %.0fx"
      % ratio_observed)
print()
print("test 2, the magnitude:")
S_needed = h2_step(100) / (L31["gain100_spread_pct"] / 100)
print("  for the step to be 1.79%% of a rung, that rung must sit at %.1f counts" % S_needed)
for S in (500, 2000, 3500):
    print("  at %5d counts the step is %.4f%% of the rung" % (S, 100 * h2_step(100) / S))

## 7. The dark bound: the reason given for it was wrong

`dark_current_bound` is published as an upper bound, with a reason attached: *"what limits it is the
offset state, not statistics."* That sentence had never been tested. Two readings of it make
different predictions, and session 03's own published table separates them.

**Random hopping is refuted by session 03's own numbers.** If frames inside a block landed in
different states, blocks would scatter within an exposure group and the rejection would have fired.
`n_anomalous` is **0 in every one of the fourteen blocks**, and the seven 300 s blocks agree with
each other to about 0.008 counts. There is no hopping inside a group.

**What survives is an offset that tracks exposure length** - and a peer-group rule is blind to it
*by construction*, because a peer group is frames sharing a setting and exposure is a setting. Look
at the excess column: the 300 s blocks sit at about -0.22 counts and the 600 s blocks at about
+0.64. That is a step of roughly 0.85 counts between two exposure groups, against an H2 step at
gain 250 of 0.99. It looks exactly like one state boundary crossed between two exposures.

**The conditional number, and why it stays conditional.** Allow each exposure group an integer
number of H2 steps - integers only, and the step is predicted rather than fitted - and the
assignment `(0, 0, 1, 1)` linearises all fourteen blocks to 0.016 counts rms, which is the
within-group noise floor. The dark current implied is 1.44e-3 e-/px/s.

That is **2.7 times session 03's published upper bound** - which is the clearest possible sign that
it is not yet a measurement. Slope, intercept and three relative integers against four exposure
groups is under-determined, and the fit reaching the noise floor is what an under-determined fit
does. **The decisive test is a fifth and sixth exposure**, which is a bench night and not a
re-analysis.

**And it changes no capture setting either way.** At L32's sky rate, the conditional dark current
is 0.09% of the sky signal. This section is about the honesty of the record, not about the model.

In [ ]:
g250 = G_PUB[250]
step250 = h2_step(250)
groups = sorted(blocks.exptime.unique())

print("session 03's fourteen blocks, by exposure group:")
for gr in groups:
    m = blocks.exptime == gr
    print("  %6.1f s   n=%2d   mean excess %+.4f   sd %.4f   anomalous rejected %d"
          % (gr, m.sum(), blocks.excess[m].mean(), blocks.excess[m].std(ddof=0),
             int(blocks.n_anomalous[m].sum())))
print()
print("H2 step at gain 250: %.4f counts" % step250)
print("300 s to 600 s jump in mean excess: %.4f counts"
      % (blocks.excess[blocks.exptime == 600].mean() - blocks.excess[blocks.exptime == 300].mean()))

ks = (0, 0, 1, 1)
adj = blocks.excess + np.array([ks[groups.index(t)] for t in blocks.exptime]) * step250
b, a = np.polyfit(blocks.exptime, adj, 1)
rms = float(np.sqrt(np.mean((adj - (a + b * blocks.exptime)) ** 2)))

fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.2))
ax[0].plot(blocks.exptime, blocks.excess, "o", ms=6, color="0.35")
ax[0].axhline(0, color="0.6", lw=0.6)
ax[0].set(xlabel="exposure, s", ylabel="excess over pedestal, counts",
          title="as published: no straight line through it")
ax[1].plot(blocks.exptime, adj, "o", ms=6, color="crimson")
tt = np.linspace(0, 640, 50)
ax[1].plot(tt, a + b * tt, "-", lw=1.0, color="0.4")
ax[1].set(xlabel="exposure, s", ylabel="excess plus integer steps, counts",
          title="with steps (0, 0, 1, 1) allowed: rms %.4f counts" % rms)
plt.tight_layout()

print()
print("conditional D published by 11: %.3e e-/px/s"
      % K["dark_current_conditional"]["value"])
print("session 03's published upper bound: %.3e e-/px/s" % K3["dark_current_bound"]["value"])
print("the conditional value is %.1fx the bound - which is why it is conditional, not a result"
      % (K["dark_current_conditional"]["value"] / K3["dark_current_bound"]["value"]))
print()
print("what limits the bound, as now published: %s" % K["dark_bound_limited_by"]["value"])
print()
print("and what it is worth to the model, against L32's sky rate of %.3f e-/px/s:" % SKY_E_PER_S)
for D in (K3["dark_current_bound"]["value"], K["dark_current_conditional"]["value"]):
    print("  D = %.3e e-/px/s is %.3f%% of the sky" % (D, 100 * D / SKY_E_PER_S))

## 8. What is settled, and what the next session inherits

**Settled, and available to every later notebook:**

| constant | value | what it means |
|---|---|---|
| `system_gain_restated` | 9.403 to 0.0492 e-/count | `g` refit with flats and bias in the same state |
| `system_gain_move_pct` | seven of eight inside 1.344% | the state cost `g` less than the gain law's own scatter |
| `ptc_pedestal_mixture_offset` | -0.37 to +0.10 counts | how far each PTC pedestal sat above its near state |
| `ptc_bias_occupancy` | 0 to 0.2 by gain | a third independent night's occupancy, at settings two others also measured |
| `state_step_at_gain100` | 0.5129 counts | H2 confirmed out of sample, to +5.1%, through the published classifier |
| `state_explains_L31` | `false` | a named candidate removed, by two independent arithmetic tests |
| `dark_bound_limited_by` | exposure-correlated offset | session 03's stated reason corrected |
| `dark_current_conditional` | 1.44e-3 e-/px/s | what `D` would be under the best integer-step assignment, and not a result |

**One decision is open, and it is the first item for review.** `results/` now holds two different
numbers for `system_gain`: session 02's in `ptc_constants.json`, and `11`'s in
`state_repair_constants.json`. That is a trap if it is left standing and a decision if it is taken.
`11` deliberately did not take it - a notebook that overwrites its predecessor before anyone has
reviewed the correction is a worse failure than a documented disagreement. The case for leaving
`ptc_constants.json` alone is section 4: at seven of eight gains the two differ by less than the
frames' own spread, so replacing them buys precision the data does not have.

**Two `LEGACY` entries were touched and neither is drained.** L31 is narrowed - its bimodality is
published and the state is ruled out as its mechanism - so it goes to the linearity session with a
smaller question and one candidate fewer.

**What needs the camera, now with reasons rather than hunches:**

- **gain 200.** Still the one point that would put the H2 law on its own feet: it is the HCG
  boundary itself, and no session has resolved a step there.
- **a fifth and sixth dark exposure.** Section 7's assignment is under-determined with four groups.
  Two more exposure lengths make it testable rather than merely consistent.
- **protocol 04's arms should stop being duty regimes** (D72): draw the idle gap in arm B too, or
  run a block of the baseline gain at arm B's cadence, so cooler duty becomes a regressor instead of
  a passenger. It costs frames, not hours.

**What does not need the camera, and is the larger prize.** Every camera-side constant the model
consumes is now measured except `ceiling(gain)`, and three `LEGACY` entries (L09, L12, L28) are
queued waiting for exactly that. The linearity session is the next build step, and it inherits
L31 in its narrowed form as a bonus question.